# PyTorch training loop over Zarr cutouts

**Environment:** [uv](https://docs.astral.sh/uv/) project venv from the repo README (`uv venv --python 3.11 .venv`, `uv sync --extra desi --extra dev`). In Jupyter, use kernel **Python (data-lake)** (`uv run python -m ipykernel install --user --name=data-lake --display-name "Python (data-lake)"`).

Demonstrates `CutoutDataset` (map-style, for training with a shuffled sampler) and
`TileCutoutDataset` (iterable-style, for full-epoch streaming).

In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler
import numpy as np

from data_lake.io.catalog import CatalogAccessor
from data_lake.ml.dataset import CutoutDataset, TileCutoutDataset

LAKE_ROOT = Path("/data/lake")
SURVEY = "des_dr2"
BATCH_SIZE = 64
NUM_WORKERS = 4

## 1. Fetch source IDs to train on

Use the catalog to select sources meeting science criteria.

In [ ]:
cat = CatalogAccessor(LAKE_ROOT, SURVEY)

# Select galaxies with good S/N and no quality flags
df = cat.query("""
    SELECT source_id
    FROM catalog
    WHERE mag_i BETWEEN 18.0 AND 23.0
      AND flag_star = 0
      AND _cutout_index >= 0
""")
source_ids = df["source_id"].tolist()
print(f"Training set size: {len(source_ids):,}")

## 2. Map-style dataset with tile-sorted sampler

Sorting batch indices by HEALPix tile before passing them to DataLoader workers
maximises Zarr shard reuse (consecutive reads hit the same shard file).

In [ ]:
def normalise(image: np.ndarray) -> torch.Tensor:
    """Log-scale stretch + normalise each band independently."""
    image = np.log1p(np.clip(image, 0, None))
    mean = image.mean(axis=(1, 2), keepdims=True)
    std  = image.std(axis=(1, 2), keepdims=True) + 1e-6
    return torch.from_numpy(((image - mean) / std).astype(np.float32))

ds = CutoutDataset(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    source_ids=source_ids,
    transform=normalise,
    tile_sort=True,
    catalog_accessor=cat,
)

print(ds)
print("Sample image shape:", ds[0].shape)

In [ ]:
# Use tile_sorted_indices as a sampler for I/O-friendly ordered access
tile_sampler = SubsetRandomSampler(ds.tile_sorted_indices)

loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    sampler=tile_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
print(f"Batches per epoch: {len(loader):,}")

## 3. Minimal training loop

In [ ]:
import torch.nn as nn

# Toy CNN – replace with your model
n_bands = ds[0].shape[0]
model = nn.Sequential(
    nn.Conv2d(n_bands, 32, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(8),
    nn.Flatten(),
    nn.Linear(32 * 8 * 8, 128), nn.ReLU(),
    nn.Linear(128, 1),
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

N_EPOCHS = 2  # increase for real training

for epoch in range(N_EPOCHS):
    running_loss = 0.0
    for step, batch in enumerate(loader):
        images = batch.to(device)
        # Dummy target: mean pixel value (replace with real labels)
        targets = images.mean(dim=(1, 2, 3), keepdim=False).unsqueeze(1)

        optimiser.zero_grad()
        preds = model(images)
        loss = loss_fn(preds, targets)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        if step % 50 == 0:
            print(f"Epoch {epoch+1}  step {step:>5d}  loss={running_loss/(step+1):.4f}")

    print(f"Epoch {epoch+1} done, avg loss={running_loss/len(loader):.4f}")

## 4. Iterable dataset for full-epoch streaming

In [ ]:
tile_ds = TileCutoutDataset(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    transform=normalise,
    shuffle_tiles=True,   # shuffle tile order each epoch
)

tile_loader = DataLoader(
    tile_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

for step, batch in enumerate(tile_loader):
    print(f"Step {step}: batch shape {batch.shape}")
    if step >= 4:
        break